# 01 · Diseño $2^k$ con réplica única (Python)

**Semana 3 — Diseños $2^k$ y fraccionados.**

## Objetivos
- Calcular los **efectos** de un diseño $2^4$.
- Identificar los efectos importantes con el **gráfico de probabilidad normal** (réplica única, sin estimador del error).
- Construir el modelo reducido y validar supuestos.

> Teoría: [`../teoria/01-disenos-2k.md`](../teoria/01-disenos-2k.md).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
import statsmodels.api as sm

np.random.seed(3)

## 1. Los datos

Experimento de **filtración** (Montgomery, ej. 6.2): se estudia la **tasa de filtración** (gal/h) de un proceso químico con 4 factores a dos niveles, codificados $-1$/$+1$:
- $A$ = temperatura, $B$ = presión, $C$ = concentración, $D$ = velocidad de agitación.

**Una sola réplica** → 16 corridas.

In [ ]:
df = pd.read_csv('../datos/filtracion-2k.csv')
df.head()

## 2. Cálculo de efectos

Ajustamos el modelo completo $2^4$ por regresión sobre las variables codificadas. El **efecto** de cada término es el doble del coeficiente ($\beta\times2$).

In [ ]:
modelo = smf.ols('tasa ~ A*B*C*D', data=df).fit()
efectos = (modelo.params * 2).drop('Intercept')
efectos.reindex(efectos.abs().sort_values(ascending=False).index).round(2)

## 3. Gráfico de probabilidad normal de los efectos

Con réplica única **no hay grados de libertad para el error**. El método de Daniel: los efectos **despreciables** (ruido) caen sobre una recta; los **importantes** se alejan.

In [ ]:
ef = efectos.sort_values()
n = len(ef)
# cuantiles teóricos normales (posiciones de probabilidad)
q = stats.norm.ppf((np.arange(1, n+1) - 0.5) / n)
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(ef.values, q)
for nombre, val in ef.items():
    if abs(val) > 8:  # etiqueta los grandes
        ax.annotate(nombre, (val, q[list(ef.index).index(nombre)]),
                    textcoords='offset points', xytext=(5,0))
ax.axvline(0, color='gray', ls='--')
ax.set_xlabel('Efecto estimado'); ax.set_ylabel('Cuantil normal')
ax.set_title('Gráfico de probabilidad normal de efectos'); plt.show()

Los efectos **$A$, $C$, $D$, $AC$ y $AD$** se separan claramente de la recta: son los **significativos**. El resto se alinea en torno a 0 (ruido). Nótese que $B$ (presión) es despreciable.

## 4. Modelo reducido y supuestos

Reajustamos solo con los términos activos (y los principales por jerarquía).

In [ ]:
reducido = smf.ols('tasa ~ A + C + D + A:C + A:D', data=df).fit()
print(sm.stats.anova_lm(reducido).round(4))
print(f'R2 = {reducido.rsquared:.4f}')

In [ ]:
resid = reducido.resid; ajust = reducido.fittedvalues
fig, ax = plt.subplots(1, 2, figsize=(11,4))
sm.qqplot(resid, line='s', ax=ax[0]); ax[0].set_title('Q-Q de residuales')
ax[1].scatter(ajust, resid); ax[1].axhline(0, color='gray', ls='--')
ax[1].set_xlabel('Ajustados'); ax[1].set_ylabel('Residual')
ax[1].set_title('Residuales vs. ajustados')
plt.tight_layout(); plt.show()

## 5. Conclusiones
- Factores activos: **$A$ (temperatura)**, **$C$ (concentración)**, **$D$ (agitación)** y las interacciones **$AC$** y **$AD$**.
- $B$ (presión) **no** influye: puede fijarse por costo/conveniencia.
- Como hay interacciones $AC$ y $AD$, los niveles óptimos de $C$ y $D$ dependen de $A$.

> Equivalente en R: [`01-disenos-2k_r.ipynb`](01-disenos-2k_r.ipynb).